In [1]:
import numpy as np
import pandas as pd

import pygeohash as pgh

from sklearn.model_selection import KFold

In [2]:
train = pd.read_csv("../dataset/train.csv")
test = pd.read_csv("../dataset/test.csv")

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [3]:
for df in [train, test]:

    df["hour"] = (
        df["timestamp"]
        .str.split(":")
        .str[0]
        .astype(int)
    )

    df["minute"] = (
        df["timestamp"]
        .str.split(":")
        .str[1]
        .astype(int)
    )

    df["minute_of_day"] = (
        df["hour"] * 60
        + df["minute"]
    )

In [4]:
for df in [train, test]:

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

In [5]:
for df in [train, test]:

    df["geo4"] = df["geohash"].str[:4]
    df["geo5"] = df["geohash"].str[:5]
    df["geo6"] = df["geohash"].str[:6]

In [6]:
for col in ["geo4", "geo5", "geo6"]:

    freq = train[col].value_counts()

    train[f"{col}_freq"] = train[col].map(freq)

    test[f"{col}_freq"] = test[col].map(freq)

In [7]:
def decode_lat(x):
    return pgh.decode(x)[0]

def decode_lon(x):
    return pgh.decode(x)[1]

In [8]:
for df in [train, test]:

    df["lat"] = df["geohash"].apply(
        decode_lat
    )

    df["lon"] = df["geohash"].apply(
        decode_lon
    )

In [9]:
TARGET = "demand"

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [10]:
for col in ["geo4", "geo5", "geo6"]:

    train[f"{col}_mean_demand"] = np.nan

    for tr_idx, val_idx in kf.split(train):

        tr_fold = train.iloc[tr_idx]
        val_fold = train.iloc[val_idx]

        mapping = (
            tr_fold.groupby(col)[TARGET]
            .mean()
        )

        train.loc[
            train.index[val_idx],
            f"{col}_mean_demand"
        ] = (
            val_fold[col]
            .map(mapping)
        )

    global_mean = train[TARGET].mean()

    train[f"{col}_mean_demand"] = (
        train[f"{col}_mean_demand"]
        .fillna(global_mean)
    )

    mapping = (
        train.groupby(col)[TARGET]
        .mean()
    )

    test[f"{col}_mean_demand"] = (
        test[col]
        .map(mapping)
        .fillna(global_mean)
    )

In [11]:
for col in ["geo4", "geo5", "geo6"]:

    feature_name = (
        f"{col}_hour_mean"
    )

    train[feature_name] = np.nan

    for tr_idx, val_idx in kf.split(train):

        tr_fold = train.iloc[tr_idx]
        val_fold = train.iloc[val_idx]

        mapping = (
            tr_fold.groupby(
                [col, "hour"]
            )[TARGET]
            .mean()
        )

        train.loc[
            train.index[val_idx],
            feature_name
        ] = (
            val_fold.set_index(
                [col, "hour"]
            )
            .index
            .map(mapping)
        )

    global_mean = train[TARGET].mean()

    train[feature_name] = (
        train[feature_name]
        .fillna(global_mean)
    )

    mapping = (
        train.groupby(
            [col, "hour"]
        )[TARGET]
        .mean()
    )

    test[feature_name] = (
        test.set_index(
            [col, "hour"]
        )
        .index
        .map(mapping)
        .fillna(global_mean)
    )

In [12]:
train.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,geo5_freq,geo6_freq,lat,lon,geo4_mean_demand,geo5_mean_demand,geo6_mean_demand,geo4_hour_mean,geo5_hour_mean,geo6_hour_mean
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,...,1389,33,-5.484924,90.664673,0.142910,0.146484,0.035970,0.065837,0.065837,0.032530
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,...,1389,89,-5.462952,90.686646,0.138964,0.142242,0.200072,0.066115,0.066115,0.145019
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,...,842,67,-5.462952,90.708618,0.056502,0.075456,0.126477,0.023239,0.033678,0.059017
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,...,614,42,-5.462952,90.862427,0.056808,0.033490,0.014768,0.025642,0.015965,0.020986
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,...,1389,36,-5.457458,90.675659,0.142910,0.146484,0.030562,0.065837,0.065837,0.025710


In [13]:
train.to_csv(
    "../dataset/train_fe.csv",
    index=False
)

test.to_csv(
    "../dataset/test_fe.csv",
    index=False
)